# 03 - Train, evaluate and compare

**Settings:** Accelerator = GPU T4 x2 (one GPU is used; T4 is recommended over P100), Internet = On.

**Inputs** (*Add Input > Your Work > Notebooks*):
1. output of notebook 01 (`ufm-trainval`)
2. output of notebook 02 (`ufm-test`)
3. **this notebook's previous output**, if it has one: finished runs are copied and skipped, so only new
   work runs. Old baselines are always recomputed (v2: speckle filter + validation-tuned random forest).

Order: baselines B1-B4, U-Net experiments, extra random seeds, uncertainty analysis, report.
Each U-Net run takes about 1 hour on a T4.

In [ ]:
# ---- Settings -------------------------------------------------------------
GITHUB_REPO = "https://github.com/mhdnasim6-boop/urban-flood-mapping.git"   # <- edit
BRANCH = "main"
# ---------------------------------------------------------------------------
# Private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets).
# Optional: a secret named HF_TOKEN (Hugging Face read token) avoids download rate limits.
import os, subprocess
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
url = GITHUB_REPO
if secret("GITHUB_TOKEN"):
    url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
subprocess.run(["rm", "-rf", "/tmp/ufm"])
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/tmp/ufm"], check=True)
os.chdir("/tmp/ufm")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
EXPERIMENTS = ["D_full", "B5_authors_baseline", "A_raw", "B_raw_phys", "C_raw_aux",
               "E_no_coherence", "F_no_vh"]    # seed 42, full maps
SEEDS = {"D_full": [1, 2], "A_raw": [1, 2]}      # extra random seeds -> mean +- sd
EXCLUDE = ["20231201_Jubba_2"]                   # too few reference flood pixels in the published chips
RUN_BASELINES = True
EXTRA = []            # config overrides for every run, e.g. ["train.epochs=30"]

import glob, shutil
from pathlib import Path
TRAINVAL = glob.glob("/kaggle/input/**/ufm-trainval", recursive=True)[0]
TEST = glob.glob("/kaggle/input/**/ufm-test", recursive=True)[0]
RUNS = Path("/kaggle/working/runs")
RUNS.mkdir(parents=True, exist_ok=True)
for prev in glob.glob("/kaggle/input/**/runs", recursive=True):   # resume from earlier sessions
    shutil.copytree(prev, RUNS, dirs_exist_ok=True, ignore=shutil.ignore_patterns("baselines"))
print("train/val:", TRAINVAL, "\ntest:", TEST, "\nfinished runs:",
      sorted(p.parent.parent.name for p in RUNS.glob("*/test/test_metrics.csv")))

In [ ]:
!pip install -q segmentation-models-pytorch tabulate rasterio 2>/dev/null
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
if RUN_BASELINES and not (RUNS / "baselines/settings.json").exists():
    !python scripts/run_baselines.py --test-root {TEST} --trainval-root {TRAINVAL} --out {RUNS}/baselines
    !cat {RUNS}/baselines/settings.json

In [ ]:
import subprocess
jobs = [(exp, exp, [], True) for exp in EXPERIMENTS]
jobs += [(exp, f"{exp}_s{k}", [f"train.seed={k}", f"name={exp}_s{k}"], False)
         for exp, ks in SEEDS.items() for k in ks]
for cfg, name, over, maps in jobs:
    if (RUNS / name / "test" / "test_metrics.csv").exists():
        print(f"skip {name}: already done"); continue
    subprocess.run(["python", "scripts/train.py", "--config", f"configs/experiments/{cfg}.yaml",
                    "--data-root", TRAINVAL, "--out", str(RUNS), *EXTRA, *over], check=True)
    subprocess.run(["python", "scripts/evaluate.py", "--run", str(RUNS / name), "--test-root", TEST,
                    "--mc", "0", *([] if maps else ["--no-maps"])], check=True)
if not (RUNS / "D_full" / "test_mc" / "test_metrics.csv").exists():   # uncertainty maps for the proposed model
    subprocess.run(["python", "scripts/evaluate.py", "--run", str(RUNS / "D_full"), "--test-root", TEST,
                    "--mc", "20", "--tag", "mc"], check=True)

In [ ]:
!python scripts/uncertainty_analysis.py --maps {RUNS}/D_full/test_mc --test-root {TEST} --out /kaggle/working/report --exclude-events {' '.join(EXCLUDE)}

In [ ]:
!python scripts/summarize.py --runs {RUNS} --test-root {TEST} --out /kaggle/working/report --exclude-events {' '.join(EXCLUDE)}

In [ ]:
from IPython.display import Markdown, Image, display
for md in ["comparison.md", "uncertainty.md"]:
    display(Markdown(open(f"/kaggle/working/report/{md}").read()))
for p in ["f1_bars.png", "uncertainty.png"] + sorted(Path("/kaggle/working/report").glob("maps_*.png")):
    display(Image(str(Path("/kaggle/working/report") / p)))